# Answer-subspace follow-ups — donor **gemma-2-9b** -> recipient **gemma-2-2b**

Five post-hoc follow-ups (**A-E**) on **one** trained task map. Pair: donor `google/gemma-2-9b`
(42 layers, d_model 3584), recipient `google/gemma-2-2b` (26 layers, d_model 2304),
`SINGLE_PAIR = (L2_SINGLE, L9_SINGLE) = (20, 34)`. Task: `a * b + c`, 3000 train / 2000 eval,
unsolvable bin ~638.

**NAMING IS LEGACY**: the `_9b` suffix means **DONOR** and the `_2b` suffix means **RECIPIENT**
everywhere, so the shared helper cells stay byte-identical to the verified base notebooks.

### VRAM — 48 GB card (A6000 / A40 / L40S)
Donor gemma-2-9b in **bf16 ~18.5 GiB** plus recipient gemma-2-2b in bf16 ~5.2 GiB = **~24 GiB of
weights**, leaving ~24 GiB of headroom for activations at `ARITH_BATCH=16` / `DONOR_BATCH=8`.
The donor is **NOT** 4-bit: on this transformers/bitsandbytes build the 4-bit path runs the
unquantized layers in fp16, activations overflow, logits go NaN, and the argmax collapses to
token 0. The NaN health-check assert in CELL 6 catches that immediately — do not remove it.

### RUN ORDER
1. Run **CELL 1** (install). It uninstalls `torchvision`/`torchaudio` *before* anything imports
   torch, because a version mismatch makes `import transformers` crash.
2. **RESTART THE KERNEL.**
3. Run **CELL 1b** downward, in order. Do not skip CELL 8/9/10/11 — A-E all reuse their globals.

### CRITICAL STRUCTURE
The task map is trained **ONCE** (CELL 9, 5 seeds) and **reused** by all five follow-ups.
Nothing below CELL 9 retrains it. Each follow-up sits behind its own flag in CELL 2b.

| flag | what it tests |
|---|---|
| `RUN_A` | **prompt-mismatch adaptivity** — graft the donor state for `P` while the recipient reads `P'=(a,b,c+delta)`: does the stitch deliver `P`'s *stale* answer, or something the recipient adapts to the problem actually on screen? (probe-free, ablation-free) |
| `RUN_B` | **rescoring the shuffle control** — re-run shuffle *with output capture* and ask how often the recipient emits the donor's answer to the problem it never saw, against an empirical first-digit null |
| `RUN_C` | **unembedding-span ablation** — define the answer subspace from the recipient's own gain-scaled unembedding rows (probe-independent) and ablate it |
| `RUN_D` | **sufficiency split** — graft `P f(h)` ALONE (the new condition) vs `(I-P) f(h)` alone (which *is* the existing INLP ablation), at natural and norm-matched magnitude |
| `RUN_E` | **SVD ablation curve + containment** — remove the top-`j` *left* singular directions the map WRITES into, and measure how much of `U_k` lies inside the answer subspace, against the correct random null |

All numbers land in `RESULTS` and the SAVE cell writes `answersubspace_followups_results.json`.

In [1]:
# === CELL 1: install (run once, then RESTART KERNEL) ===
# torchvision/torchaudio are uninstalled BEFORE any torch import: a version mismatch between
# them and torch makes `import transformers` crash. Nothing in this cell imports torch.
!pip install -q "transformers==4.46.3" accelerate bitsandbytes numpy matplotlib datasets hf_transfer
!pip uninstall -y torchvision torchaudio
# >>> RESTART THE KERNEL NOW, before running any other cell. <<<
# (Nothing above imports torch; everything below assumes a fresh kernel.)
# --- Blackwell/sm_120 pods ONLY (CELL 1b prints cap (12,0)): uncomment, run, restart again ---
# !pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu128


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Found existing installation: torchvision 0.19.1+cu124
Uninstalling torchvision-0.19.1+cu124:
  Successfully uninstalled torchvision-0.19.1+cu124
Found existing installation: torchaudio 2.4.1+cu124
Uninstalling torchaudio-2.4.1+cu124:
  Successfully uninstalled torchaudio-2.4.1+cu124


In [1]:
# === CELL 1b: post-restart verification (run FIRST after the kernel restart) ===
import torch, transformers
print("torch:", torch.__version__, "| cuda cap:", torch.cuda.get_device_capability(0))
print("transformers:", transformers.__version__, "(want 4.46.3)")
free, total = torch.cuda.mem_get_info()
print(f"GPU: {torch.cuda.get_device_name(0)} | total {total/2**30:.1f} GiB | free {free/2**30:.1f} GiB")
assert total/2**30 > 40, ("this notebook needs a 48 GB card (A6000/A40/L40S): the gemma-2-9b donor "
                          "is ~18.5 GiB in bf16, the gemma-2-2b recipient ~5.2 GiB, and the donor "
                          "MUST NOT be quantized (see the header).")

torch: 2.4.1+cu124 | cuda cap: (8, 9)
transformers: 4.46.3 (want 4.46.3)
GPU: NVIDIA L40S | total 44.4 GiB | free 44.0 GiB


In [2]:
# === CELL 2: imports, set_submodule shim, global config (VRAM/compute switches) ===
# VERBATIM from CELL 2 of the verified base notebook apart from the model ids, the layer pair,
# and the batch sizes (see the header markdown for why).
#
# NAMING: the `_2b` suffix means RECIPIENT and the `_9b` suffix means DONOR throughout. Those
# names are legacy (the validated v1 pair was 2B<-9B) and are kept so the shared helper cells
# stay byte-identical to the source notebooks. Here: DONOR = gemma-2-9b, RECIPIENT = gemma-2-2b.
import os, json, math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# shim: newer transformers' 4-bit path calls nn.Module.set_submodule, absent on older torch
if not hasattr(nn.Module, "set_submodule"):
    def _set_submodule(self, target, module):
        mod = self
        atoms = target.split(".")
        for a in atoms[:-1]:
            mod = getattr(mod, a)
        setattr(mod, atoms[-1], module)
    nn.Module.set_submodule = _set_submodule

DEVICE = "cuda"
torch.manual_seed(0)
MODEL_2B = "google/gemma-2-2b"    # RECIPIENT (26 layers, d_model 2304)
MODEL_9B = "google/gemma-2-9b"    # DONOR     (42 layers, d_model 3584, bf16 ~18.5 GB)

# Smoke test lever (True for a ~10 min shakedown, False for the real run)
SMOKE_TEST = False

# VALIDATED layer pair for this exact pair — already derived by EXP1 in the base notebook.
# Do NOT re-derive here; the follow-ups are post-hoc on the validated site.
SINGLE_PAIR = (20, 34)
LAYER_PAIRS = [(18, 31), (20, 34), (22, 37), (24, 40)]
L2_SINGLE, L9_SINGLE = SINGLE_PAIR
PATCH_POS = -1
RIDGE_LAMBDA = 1e3

if SMOKE_TEST:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 12, 200, 120
    GSM_FIT, GSM_EVAL, GSM_STRENGTHS = 20, 24, [1.0]
    TASK_SEEDS, TASK_EPOCHS = [0, 1], 2
    BOOT_B = 1000
else:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 300, 3000, 2000   # 3000/2000 matches v1 exactly
    GSM_FIT, GSM_EVAL, GSM_STRENGTHS = 150, 400, [1.0]
    TASK_SEEDS, TASK_EPOCHS = [0, 1, 2, 3, 4], 6             # 5 seeds for the task map
    BOOT_B = 10000

# BATCH SIZES for a 48 GB card: 9B donor + 2B recipient ~= 24 GiB of resident weights, so
# ARITH_BATCH 16 and GSM_BATCH 8 leave ample activation headroom (peak observed ~30-34 GiB).
ARITH_BATCH, GSM_BATCH, MAX_NEW_GSM, MAX_NEW_ARITH = 16, 8, 300, 8
RESULTS = {}   # everything defensible gets concentrated here and printed at the end
print("SMOKE_TEST =", SMOKE_TEST, "| recipient:", MODEL_2B, "| donor:", MODEL_9B)
print("validated pair: recipient L%d <- donor L%d" % (L2_SINGLE, L9_SINGLE))

SMOKE_TEST = False | recipient: google/gemma-2-2b | donor: google/gemma-2-9b
validated pair: recipient L20 <- donor L34


In [3]:
# === CELL 2b: experiment flags + follow-up knobs (run right after CELL 2) ===
# The base pipeline (CELL 8/9/10) is NOT optional: A-E are post-hoc on the ONE map it trains.
RUN_CORE = True   # base channel numbers (headline conferral ~0.89) — the reference every
                  # follow-up is compared against. Leave True unless you are re-running.
RUN_INLP = True   # the INLP answer subspace A. REQUIRED by C (comparison), D (P = A) and E
                  # (overlay + containment). Turning it off degrades C/D/E, it does not skip them.

# ---- the five follow-ups, one flag each ----
RUN_A = True      # A. prompt-mismatch adaptivity ("off-by-one")            HIGHEST PRIORITY
RUN_B = True      # B. shuffle rescored for stale-answer emission (re-run WITH output capture)
RUN_C = True      # C. unembedding-span ablation (probe-independent answer subspace)
RUN_D = True      # D. sufficiency split: P f(h) alone vs (I-P) f(h) alone
RUN_E = True      # E. SVD ablation curve + containment of the map's write directions

# ---- INLP (answer subspace A) ----
INLP_ROUNDS       = 90     # each round adds <= 9 directions; 90 rounds reaches rank ~540+
INLP_CHECK_EVERY  = 4      # measure conferral (answer-erased + matched random) every N rounds
INLP_TARGET_RANKS = [360, 540]   # exact-rank snapshots kept for D; 540 = "converged" in v1
COLLAPSE_FRAC     = 0.5

# ---- A: prompt mismatch ----
A_MAX_ABS_DELTA   = 98     # c' = c + delta must stay in [1, 99] (the generator's own range)
A_FULL_ANSWER     = True   # also classify the free generation, not just the first token
A_SHUFFLE_SEED    = 1234

# ---- B: shuffle rescoring ----
B_NULL_DRAWS      = 20     # empirical first-digit null: pair each generation with a THIRD problem
B_SHUFFLE_SEED    = 7      # derangement seed for the donor->recipient mismatch

# ---- C: unembedding span ----
C_RANDOM_SEEDS    = 20     # matched-rank random control, mean +/- CI across seeds

# ---- D: sufficiency split ----
D_SUBSPACE_RANKS  = [540, 360]   # INLP ranks to split at; the ~9-dim B from C is added too
D_FULL_ANSWER     = True         # free-generation full-answer conferral as well as first token

# ---- E: SVD ablation + containment ----
E_JS              = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
E_RANDOM_SEEDS    = 5      # matched-rank random ablation at each j
E_CONTAIN_KS      = [8, 16, 32]
E_CONTAIN_NULL_SEEDS = 20  # empirical check of the analytic null dim(subspace)/d

OUT_JSON = "answersubspace_followups_results.json"
print("flags:", {k: v for k, v in sorted(globals().items())
                 if k.startswith("RUN_") and isinstance(v, bool)})
print("out:", OUT_JSON)

flags: {'RUN_A': True, 'RUN_B': True, 'RUN_C': True, 'RUN_CORE': True, 'RUN_D': True, 'RUN_E': True, 'RUN_INLP': True}
out: answersubspace_followups_results.json


In [ ]:
# === CELL 3: Hugging Face login (Gemma is gated) ===
# PLACEHOLDER — paste your own read token here. Never commit a real token.
from huggingface_hub import login
login("")

In [6]:
# === CELL 4: statistics helpers (match the interval to the source of randomness) ===
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return p, max(0.0, c-h), min(1.0, c+h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
def bootstrap_ci(x, B=None, alpha=0.05, seed=0):
    x = np.asarray(x, float); n = len(x); B = B or BOOT_B
    if n == 0: return (float("nan"),)*3
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, n, (B, n))].mean(1)
    lo, hi = np.percentile(m, [100*alpha/2, 100*(1-alpha/2)])
    return float(x.mean()), float(lo), float(hi)
_TC = {2:12.706,3:4.303,4:3.182,5:2.776,6:2.571,7:2.447,8:2.365,9:2.306,10:2.262}
def across_seed_ci(v, alpha=0.05):
    v = np.asarray(v, float); k = len(v); m = float(v.mean())
    if k < 2: return (m, float("nan"), float("nan"))
    se = v.std(ddof=1)/math.sqrt(k); t = _TC.get(k, 1.96)
    return (m, m-t*se, m+t*se)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"

In [7]:
# === CELL 5: shared helpers (hooks, padding, ridge map, problems, GSM8K, full-answer) ===
def _hid(o): return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,)+tuple(o[1:]) if isinstance(o, tuple) else h
def capture(store, key):
    def hook(_m,_i,o): store[key] = _hid(o)[:, PATCH_POS, :].detach()
    return hook
def patch_vec(vec):  # replace last-pos with vec (graph-safe: works under autograd too)
    def hook(_m,_i,o):
        h = _hid(o)
        if h.shape[1] == 0: return o
        h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
        return _pack(o, h2)
    return hook

def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L-t.numel():] = t; m[i, L-t.numel():] = 1
    return ids, m

def fit_ridge(X9, X2, lam=RIDGE_LAMBDA):
    mu9, mu2 = X9.mean(0), X2.mean(0)
    A, B = X9-mu9, X2-mu2
    W = torch.linalg.solve(A.T@A + lam*torch.eye(A.shape[1]), A.T@B)
    return mu9, mu2, W
def apply_map(x, m): mu9, mu2, W = m; return (x-mu9)@W + mu2

# ---- arithmetic problems (muladd only: healthy unsolvable bin) ----
FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n"
           "3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(tok, expr, ans):
    # Build prompt and prompt+answer, then target the FIRST answer token that carries a
    # digit. On Gemma the answer tokenizes as [space, digit] so that token is at len(p)+1;
    # on byte-level BPE tokenizers (Qwen, Llama) the leading space fuses with the first
    # digit, so it's at len(p). Scanning for the first digit-bearing token handles both,
    # plus any tokenizer that emits leading whitespace/markup tokens before the number.
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]   # context up to (not incl.) the first digit token; target = that token
def gen_arith(tok, n, rng, exclude=None):
    exclude = exclude or set(); out, seen = [], set()
    tries = 0
    while len(out) < n and tries < n*120:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids, tok_id = _aencode(tok, expr, ans)
        if tok_id is None: continue
        out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
    return out

# ---- GSM8K: validated prompt + extraction (from the Linear_gsm8k notebook) ----
import re as _re
def gsm_prompt(q):
    return (
        "Below are math problems with detailed step-by-step solutions.\n\n"
        "Problem: Natalia sold clips to 48 of her friends in April, and then she sold "
        "half as many clips in May. How many clips did Natalia sell altogether in April and May?\n"
        "Solution: Let's think step-by-step.\n"
        "1. Clips sold in April: 48\n"
        "2. Clips sold in May: 48 / 2 = 24\n"
        "3. Total clips: 48 + 24 = 72\n"
        "#### 72\n\n"
        f"Problem: {q}\n"
        "Solution: Let's think step-by-step."
    )
def gsm_extract(text):
    m = _re.search(r"####\s*(-?[\d,.]+)", text)
    if m:
        try: return float(m.group(1).replace(",", ""))
        except ValueError: pass
    m = _re.search(r"answer is\s*(-?[\d,.]+)", text, _re.IGNORECASE)
    if m:
        try: return float(m.group(1).replace(",", ""))
        except ValueError: pass
    nums = _re.findall(r"-?[\d,.]+", text)
    if nums:
        try: return float(nums[-1].rstrip(".").replace(",", ""))
        except ValueError: return None
    return None
def _parse_first_int(text):
    """Arithmetic answers: take the FIRST integer the model emits after '=',
    not the last (the model may continue with few-shot-style lines)."""
    m = _re.search(r"-?\d+", text)
    return float(m.group()) if m else None

# ---- generic batched forward: last-pos resid at given layers + top token ----
@torch.inference_mode()
def states_and_top(model, layers, prob_ids, batch=ARITH_BATCH):
    base = model.model if hasattr(model, "model") else model
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return {L: torch.cat(v) for L, v in acc.items()}, top

# ---- per-batch graft hook: replace last prompt-position resid with vec[B,d] ----
# Fires only during prefill (seq len > 1); no-ops during generation (len==1) and
# when the batch dim doesn't match, so generation proceeds normally after seeding.
_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]:
        return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)

@torch.inference_mode()
def arith_fullanswer_correct(model, layer, probs, vecs=None, batch=ARITH_BATCH):
    """Generate the full number and compare to gold. If vecs is given, vecs[i] is
    grafted at the last prompt position of problem i (prefill) before generation.
    Returns list[bool], one per problem."""
    ok, handle = [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=MAX_NEW_ARITH,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                ok.append(pred is not None and abs(pred - p["ans"]) < 0.5)
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return ok

print("helpers defined")

helpers defined


In [8]:
# === CELL 5b: extra helpers used by the experiment cells (probes, digits, datasets) ===
# fd() is defined HERE on purpose: the base notebooks only define it inside a late experiment
# cell, and several cells below call it earlier. Everything an experiment cell needs that is
# not in CELL 4 / CELL 5 lives in this cell, so there is exactly one place to look.
import collections
from datasets import load_dataset

def fd(x):
    """Leading (first) digit of an integer answer, 0-9."""
    return int(str(abs(int(round(x))))[0]) if x is not None else 0

def _dig(ans, k):
    """k-th digit (1-indexed from the left) of an integer answer, or None if too short."""
    s = str(abs(int(ans)))
    return int(s[k-1]) if len(s) >= k else None

def fit_probe_w(X, y, nclass=10, steps=300, lr=1e-2):
    """Multinomial linear probe by full-batch Adam. Returns (weights, feature mean).
    Runs on whatever device X is on, so it serves both the CPU state probes and the
    on-DEVICE INLP loop."""
    dev = X.device
    Pw = torch.zeros(X.shape[1], nclass, device=dev, requires_grad=True)
    opt = torch.optim.Adam([Pw], lr=lr)
    mu = X.mean(0).detach(); Xc = (X - mu).detach(); yy = y.to(dev)
    for _ in range(steps):
        opt.zero_grad(); F.cross_entropy(Xc @ Pw, yy).backward(); opt.step()
    return Pw.detach(), mu

def probe_first_digit(Xtr, ytr, Xte, yte=None, steps=300):
    """Fit on (Xtr,ytr), return PREDICTIONS on Xte (cpu long tensor). yte is accepted and
    ignored so the call signature matches the base notebooks verbatim."""
    Pw, mu = fit_probe_w(Xtr, ytr, steps=steps)
    return ((Xte.to(Pw.device) - mu) @ Pw).argmax(1).cpu()

def probe_split_bools(X, y, steps=300):
    """Half/half split of one state matrix -> list[bool] of test-half correctness."""
    h = len(y) // 2
    pred = probe_first_digit(X[:h], y[:h], X[h:], steps=steps)
    return (pred == y[h:].cpu()).tolist()

def majority_acc(y):
    y = torch.as_tensor(y)
    if y.numel() == 0: return float("nan")
    maj = collections.Counter(y.tolist()).most_common(1)[0][0]
    return round(float((y == maj).float().mean()), 3)

print("extra helpers defined (fd, _dig, fit_probe_w, probe_first_digit, probe_split_bools, majority_acc)")

extra helpers defined (fd, _dig, fit_probe_w, probe_first_digit, probe_split_bools, majority_acc)


In [9]:
# === CELL 6: load both models once; donor in bf16 (NOT 4-bit) ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_2B)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

# QUANTIZE_9B: under 4-bit this transformers/bnb build runs the UNQUANTIZED layers in FP16,
# Gemma-2 activations overflow fp16 (>65504) -> NaN logits -> argmax collapses to token 0 ('!').
# bf16 has the exponent range to avoid this. Donor state fidelity AT THE GRAFT SITE is the whole
# object of study here, so the donor stays bf16. On 48 GB it fits with ~24 GiB to spare.
QUANTIZE_9B = globals().get("QUANTIZE_9B", False)
assert QUANTIZE_9B is False, "the donor must be loaded in bf16 — see the comment above"

model_2b = AutoModelForCausalLM.from_pretrained(
    MODEL_2B, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
if QUANTIZE_9B:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, quantization_config=bnb, device_map={"": 0},
        torch_dtype=torch.bfloat16, attn_implementation="eager", low_cpu_mem_usage=True).eval()
else:
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, torch_dtype=torch.bfloat16, attn_implementation="eager",
        low_cpu_mem_usage=True).to(DEVICE).eval()
print("loaded:", model_2b.config.num_hidden_layers, "x recipient layers,",
      model_9b.config.num_hidden_layers, "x donor layers |",
      "donor quantized" if QUANTIZE_9B else "donor bf16")

# Health check: catch a NaN/overflow blowup (the fp16-under-4bit failure) immediately, not
# 168 silently-filtered pairs later. A healthy donor tops a real word here, never token 0.
with torch.inference_mode():
    _hl = model_9b(tokenizer("The capital of France is", return_tensors="pt").to(DEVICE).input_ids).logits[0, -1, :]
assert not torch.isnan(_hl).any() and not torch.isinf(_hl).any(), (
    "donor produced NaN/Inf logits — numerical blowup. If QUANTIZE_9B=True, the donor is overflowing "
    "fp16; set QUANTIZE_9B=False to load it in bf16 (needs the VRAM but is numerically safe).")
del _hl

# Same-family requirement: the two models must share the TOKENIZER so positions align
# (the whole stitch grafts by position). NOTE: config.vocab_size is the *padded embedding*
# count, not the tokenizer — Qwen pads differently across sizes (0.5B=151936, 7B=152064)
# while sharing one tokenizer, so comparing config.vocab_size gives false alarms. Verify the
# tokenizer itself instead, by checking a probe string maps to identical ids under each model's
# own tokenizer. (We load one shared tokenizer, but this also catches an accidental mismatch.)
_tk2 = AutoTokenizer.from_pretrained(MODEL_2B); _tk9 = AutoTokenizer.from_pretrained(MODEL_9B)
_probe = "3 * 12 + 7 = 43\nThe answer is 256."
assert _tk2(_probe).input_ids == _tk9(_probe).input_ids, (
    "tokenizer mismatch: the two models tokenize the same text differently, so positions won't "
    "align. This notebook requires a SAME-FAMILY pair sharing one tokenizer.")
del _tk2, _tk9

D_RECIP = model_2b.config.hidden_size
D_DONOR = model_9b.config.hidden_size
NL_RECIP = model_2b.config.num_hidden_layers
NL_DONOR = model_9b.config.num_hidden_layers
DONOR_BATCH = 8   # donor forwards use THIS, never ARITH_BATCH, so left-padding (and hence the
                  # bf16 numerics, and hence the donor-solved list) is reproducible run to run.
assert 0 <= L2_SINGLE < NL_RECIP and 0 <= L9_SINGLE < NL_DONOR, "layer pair out of range"
print(f"d_model: recipient {D_RECIP}, donor {D_DONOR} | layers: recipient {NL_RECIP}, donor {NL_DONOR}")
print(f"VRAM in use: {torch.cuda.memory_allocated()/2**30:.1f} GiB allocated, "
      f"{torch.cuda.mem_get_info()[0]/2**30:.1f} GiB free")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

d_model: recipient 2304, donor 3584 | layers: recipient 26, donor 42
VRAM in use: 22.1 GiB allocated, 21.6 GiB free


In [10]:
# === CELL 8: EXP2 setup — arithmetic states, native correctness, reconstruction map ===
# Generator, seeds and n match the validated base run exactly (3000 train / 2000 eval, a*b+c).
# Donor forwards use DONOR_BATCH (NOT ARITH_BATCH) so left-padding — and therefore the bf16
# numerics and the donor-solved list — are reproducible.
train = gen_arith(tokenizer, N_ARITH_TRAIN, random.Random(0))
evalp = gen_arith(tokenizer, N_ARITH_EVAL, random.Random(1), {p["expr"] for p in train})
print(f"arith: {len(train)} train, {len(evalp)} eval")
N_EVAL_ALL = len(evalp)

X9t, _ = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in train], batch=DONOR_BATCH); X9t = X9t[L9_SINGLE]
X9e, nine_top = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in evalp], batch=DONOR_BATCH); X9e = X9e[L9_SINGLE]
keep = [i for i in range(len(evalp)) if nine_top[i] == evalp[i]["tok"]]
DONOR_SOLVE_RATE = len(keep) / max(N_EVAL_ALL, 1)
DONOR_SOLVED_EXPRS = [evalp[i]["expr"] for i in keep]
evalp = [evalp[i] for i in keep]; X9e = X9e[keep]
print(f"  donor solves {len(evalp)}/{N_EVAL_ALL} = {DONOR_SOLVE_RATE:.3f} (this is P(donor solves in one pass))")

X2t, _ = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in train]); X2t = X2t[L2_SINGLE]
X2e, two_top = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in evalp]); X2e = X2e[L2_SINGLE]
solvable = [two_top[i] == evalp[i]["tok"] for i in range(len(evalp))]
unsolv = [i for i in range(len(evalp)) if not solvable[i]]
solv = [i for i in range(len(evalp)) if solvable[i]]
print(f"  recipient natively solves {len(solv)}/{len(evalp)} (UNSOLVABLE BIN n={len(unsolv)})")

mu9, mu2, Wr = fit_ridge(X9t, X2t)
recon_map = (mu9.to(DEVICE), mu2.to(DEVICE), Wr.to(DEVICE))
def map_recon(x9): m9, m2, W = recon_map; return (x9.to(DEVICE)-m9)@W + m2

RESULTS["bins"] = {
    "n_train": len(train), "n_eval_generated": N_EVAL_ALL,
    "donor_solve_rate": round(DONOR_SOLVE_RATE, 4), "n_donor_solved": len(evalp),
    "n_recipient_solvable": len(solv), "n_unsolvable_bin": len(unsolv),
    "recipient_native_rate_on_donor_solved": round(len(solv)/max(len(evalp),1), 4)}
print("bins:", json.dumps(RESULTS["bins"], indent=2))

arith: 3000 train, 2000 eval
  donor solves 1693/2000 = 0.847 (this is P(donor solves in one pass))
  recipient natively solves 1059/1693 (UNSOLVABLE BIN n=634)
bins: {
  "n_train": 3000,
  "n_eval_generated": 2000,
  "donor_solve_rate": 0.8465,
  "n_donor_solved": 1693,
  "n_recipient_solvable": 1059,
  "n_unsolvable_bin": 634,
  "recipient_native_rate_on_donor_solved": 0.6255
}


In [11]:
# === CELL 9: EXP3 — train the task-supervised map ONCE, 5 seeds (the only stochastic part) ===
# *** THIS IS THE ONLY TRAINING IN THE NOTEBOOK. *** Follow-ups A-E are all post-hoc on the map
# produced here; none of them retrains anything.
mu9d, mu2d = mu9.to(DEVICE), mu2.to(DEVICE)
task_maps = []
model_2b.requires_grad_(False)
for seed in TASK_SEEDS:
    torch.manual_seed(seed); random.seed(seed)
    W = Wr.clone().to(DEVICE).requires_grad_(True); b = mu2.clone().to(DEVICE).requires_grad_(True)
    opt = torch.optim.Adam([W, b], lr=1e-3)
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    idx = list(range(len(train)))
    try:
        for ep in range(TASK_EPOCHS):
            random.Random(seed*100+ep).shuffle(idx)
            for s in range(0, len(idx), ARITH_BATCH):
                sub = idx[s:s+ARITH_BATCH]
                x9 = X9t[sub].to(DEVICE)
                _graft["vec"] = (x9 - mu9d) @ W + b
                ids, m = left_pad([train[k]["ids"] for k in sub], tokenizer.pad_token_id)
                lg = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([train[k]["tok"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(lg, tgt); opt.zero_grad(); loss.backward(); opt.step()
    finally:
        handle.remove(); _graft["vec"] = None
    task_maps.append((W.detach(), b.detach()))
    print(f"  seed {seed}: final batch CE {loss.item():.3f}", flush=True)
model_2b.requires_grad_(True)
print("task map trained ONCE;", len(task_maps), "seeds available. Follow-ups reuse task_maps[0].")

  seed 0: final batch CE 0.009
  seed 1: final batch CE 0.274
  seed 2: final batch CE 0.008
  seed 3: final batch CE 0.042
  seed 4: final batch CE 0.048
task map trained ONCE; 5 seeds available. Follow-ups reuse task_maps[0].


In [12]:
# === CELL 10: EXP2+3 eval — the REFERENCE numbers every follow-up is compared against ===
@torch.inference_mode()
def first_token_confer(map_fn, idxs):
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    out = []
    try:
        for i in range(0, len(idxs), ARITH_BATCH):
            sub = idxs[i:i+ARITH_BATCH]
            _graft["vec"] = map_fn(X9e[sub])
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == evalp[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        handle.remove(); _graft["vec"] = None
    return out
def full_confer(map_fn, idxs):
    vecs = list(map_fn(X9e[idxs]).cpu())
    return arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in idxs], vecs=vecs)

def map_task(i):
    W, b = task_maps[i]
    return lambda x9: (x9.to(DEVICE)-mu9d)@W + b
shuf = torch.randperm(len(evalp))
@torch.inference_mode()
def shuffle_confer(idxs):  # task map fed the WRONG problem's donor state (specificity control)
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch); out = []
    W, b = task_maps[0]
    try:
        for i in range(0, len(idxs), ARITH_BATCH):
            sub = idxs[i:i+ARITH_BATCH]
            donor = X9e[shuf[i:i+len(sub)]].to(DEVICE)
            _graft["vec"] = (donor - mu9d)@W + b
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == evalp[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        handle.remove(); _graft["vec"] = None
    return out

# NOTE: the four helpers above are defined UNCONDITIONALLY (outside the RUN_CORE guard) because
# every follow-up cell calls them. Only the measurements below are behind the flag.
if RUN_CORE:
    # recon_cos (continuous metric -> bootstrap CI); task map should sit much lower
    rc_recon = F.cosine_similarity(map_recon(X9e).cpu(), X2e, dim=1).numpy()
    rc_task = F.cosine_similarity(((X9e.to(DEVICE)-mu9d)@task_maps[0][0]+task_maps[0][1]).cpu(), X2e, dim=1).numpy()
    arr = {"recon_cos_recon": fmt(bootstrap_ci(rc_recon)),
           "recon_cos_task": fmt(bootstrap_ci(rc_task))}

    # headline bin: UNSOLVABLE — full-answer + the 5-seed treatment live here
    if unsolv:
        arr["native_unsolv_full"] = fmt(wilson_bools(arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in unsolv], vecs=None)))
        arr["recon_unsolv_full"] = fmt(wilson_bools(full_confer(map_recon, unsolv)))
        arr["recon_unsolv_first"] = fmt(wilson_bools(first_token_confer(map_recon, unsolv)))
        seed_full = [float(np.mean(full_confer(map_task(s), unsolv))) for s in range(len(task_maps))]
        arr["task_unsolv_full_acrossseed"] = fmt(across_seed_ci(seed_full))
        arr["task_unsolv_full_per_seed"] = [round(v, 4) for v in seed_full]
        arr["task_unsolv_first_pooled"] = fmt(wilson_bools(first_token_confer(map_task(0), unsolv)))
        arr["shuffle_unsolv_first"] = fmt(wilson_bools(shuffle_confer(unsolv)))
    # sanity bin: SOLVABLE — first-token + full-answer (recon and task seed 0)
    if solv:
        arr["native_solv_full"] = fmt(wilson_bools(arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in solv], vecs=None)))
        arr["recon_solv_first"] = fmt(wilson_bools(first_token_confer(map_recon, solv)))
        arr["recon_solv_full"] = fmt(wilson_bools(full_confer(map_recon, solv)))
        arr["task_solv_first"] = fmt(wilson_bools(first_token_confer(map_task(0), solv)))
        arr["task_solv_full"] = fmt(wilson_bools(full_confer(map_task(0), solv)))
    RESULTS["arithmetic"] = arr
    print("EXP2/3:", json.dumps(arr, indent=2))
else:
    print("core-channel eval skipped (RUN_CORE=False); map_task/first_token_confer/full_confer "
          "are still defined for the follow-up cells.")

EXP2/3: {
  "recon_cos_recon": "0.981 [0.980, 0.982]",
  "recon_cos_task": "0.197 [0.194, 0.200]",
  "native_unsolv_full": "0.000 [0.000, 0.006]",
  "recon_unsolv_full": "0.032 [0.021, 0.048]",
  "recon_unsolv_first": "0.188 [0.159, 0.220]",
  "task_unsolv_full_acrossseed": "0.125 [0.117, 0.134]",
  "task_unsolv_full_per_seed": [
    0.1278,
    0.1262,
    0.123,
    0.1151,
    0.1341
  ],
  "task_unsolv_first_pooled": "0.904 [0.878, 0.924]",
  "shuffle_unsolv_first": "0.109 [0.087, 0.135]",
  "native_solv_full": "0.126 [0.107, 0.147]",
  "recon_solv_first": "0.928 [0.911, 0.942]",
  "recon_solv_full": "0.120 [0.102, 0.141]",
  "task_solv_first": "0.950 [0.935, 0.962]",
  "task_solv_full": "0.101 [0.084, 0.121]"
}


In [13]:
# === CELL 10b: SHARED CONVENTIONS for the five follow-ups (A-E) — run before A ===
# Every follow-up below is POST-HOC on the ONE task map trained in CELL 9. Nothing is retrained.
#
#   h     = donor residual state at L9_SINGLE, LAST PROMPT TOKEN      (the rows of X9t / X9e)
#   f     = the trained linear task map (CE on the ground-truth first answer token)
#   f(h)  = W h + b = THE VECTOR WRITTEN INTO THE RECIPIENT at L2_SINGLE.
#           Every ablation below acts on this FINAL AFFINE OUTPUT, never on W h alone.
#
# EVERY ablation is run TWO ways and both are reported:
#   zero-ablation:  v <- (I - P P^T) v
#   mean-ablation:  v <- (I - P P^T) v + P P^T mu_native
# where mu_native is the mean of the RECIPIENT's NATIVE (unstitched) L2_SINGLE states on the
# SAME problems we score on (the unsolvable bin). Zeroing puts the residual off-distribution;
# mean-ablation keeps it on-distribution. If the two agree the result is robust; if they
# diverge, THE MEAN-ABLATION NUMBER IS THE ONE TO REPORT.

MAP_SEED = 0    # every follow-up uses task_maps[MAP_SEED] -> all A-E numbers are SINGLE-SEED.
SINGLE_SEED_NOTE = ("SINGLE SEED (task_maps[%d]); the 5-seed spread for the same quantity is in "
                    "RESULTS['arithmetic']['task_unsolv_full_per_seed']" % MAP_SEED)

def write_vec(x9, seed=MAP_SEED):
    """f(h) = W h + b — the vector actually written into the recipient at L2_SINGLE."""
    W, b = task_maps[seed]
    return (x9.to(DEVICE) - mu9d) @ W + b

# mu_native for the bin we score on (the recipient's own unstitched state at the graft site)
MU_NATIVE = X2e[unsolv].mean(0).to(DEVICE) if len(unsolv) else X2e.mean(0).to(DEVICE)

# ---- subspace algebra (Q always has ORTHONORMAL COLUMNS, shape (d_recip, r)) ----
def _proj(v, Q):
    """P P^T v for row-vectors v of shape (..., d)."""
    return (v @ Q) @ Q.T
def ablate_zero(v, Q):
    return v - _proj(v, Q)
def ablate_mean(v, Q, mu=None):
    mu = MU_NATIVE if mu is None else mu
    return v - _proj(v, Q) + _proj(mu.unsqueeze(0), Q)
def project_only(v, Q):
    """P f(h) ALONE, everything else ZEROED (the sufficiency condition of EXP D)."""
    return _proj(v, Q)
def project_only_meanfill(v, Q, mu=None):
    """P f(h) with the COMPLEMENT filled from mu_native instead of zeros (on-distribution)."""
    mu = MU_NATIVE if mu is None else mu
    m = mu.unsqueeze(0)
    return _proj(v, Q) + (m - _proj(m, Q))
def rescale_to(u, v, eps=1e-8):
    """Rescale each row of u to the norm of the corresponding row of v."""
    return u * (v.norm(dim=-1, keepdim=True) / (u.norm(dim=-1, keepdim=True) + eps))

def orth_basis(M, tol=1e-6):
    """Orthonormal basis for the COLUMN space of M (d, k). Returns (d, rank)."""
    U, S, _ = torch.linalg.svd(M.float(), full_matrices=False)
    return U[:, S > S.max() * tol].contiguous()
def rand_orth(d, r, seed=0, device=None):
    """A uniformly random r-dim orthonormal frame in R^d (matched-rank random control)."""
    device = DEVICE if device is None else device
    g = torch.Generator(device="cpu").manual_seed(int(seed))
    return torch.linalg.qr(torch.randn(d, r, generator=g).to(device)).Q[:, :r].contiguous()
def principal_cos(Qa, Qb):
    """cos(theta_i) for the principal angles between span(Qa) and span(Qb)."""
    return torch.linalg.svdvals(Qa.float().T @ Qb.float()).clamp(0.0, 1.0)
def containment(Qa, Qb):
    """(1/dim Qa) * sum_i cos^2(theta_i): the fraction of span(Qa) lying inside span(Qb).
    NULL for a random Qa: dim(span Qb)/d  — always report against THAT, not against 0."""
    c = principal_cos(Qa, Qb)
    return float((c ** 2).sum().item() / Qa.shape[1])
def angles_deg(Qa, Qb, k=None):
    c = principal_cos(Qa, Qb)
    if k is not None: c = c[:k]
    return [round(math.degrees(math.acos(min(1.0, max(0.0, float(x))))), 2) for x in c]

# ---- grafting on an ARBITRARY problem list (A and B need prompts that are not evalp[i]) ----
@torch.inference_mode()
def graft_first_tokens(probs, vecs=None, batch=ARITH_BATCH):
    """Greedy FIRST token id per problem. vecs[i] (if given) overwrites the LAST PROMPT
    position of probs[i] at L2_SINGLE during PREFILL only; vecs=None -> native, no hook."""
    handle = (model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
              if vecs is not None else None)
    tops = []
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            if vecs is not None:
                v = vecs[i:i+len(chunk)]
                _graft["vec"] = (v if torch.is_tensor(v) else torch.stack(list(v))).to(DEVICE)
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            tops += model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)
                             ).logits[:, -1, :].argmax(-1).cpu().tolist()
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return tops

@torch.inference_mode()
def graft_generations(probs, vecs=None, batch=ARITH_BATCH):
    """Same graft, greedy FREE generation. Returns (texts, first_ints) — first_ints[i] is the
    first integer the recipient emits, or None. THIS is what CELL B needs and what the base
    shuffle_confer threw away (it returned bools only)."""
    handle = (model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
              if vecs is not None else None)
    texts = []
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            if vecs is not None:
                v = vecs[i:i+len(chunk)]
                _graft["vec"] = (v if torch.is_tensor(v) else torch.stack(list(v))).to(DEVICE)
            gen = model_2b.generate(ids, attention_mask=m, max_new_tokens=MAX_NEW_ARITH,
                                    do_sample=False, pad_token_id=tokenizer.eos_token_id)
            texts += tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return texts, [_parse_first_int(t) for t in texts]

# ---- the workhorse for C/D/E: T transforms f(h) into the vector actually grafted ----
# no_grad, not inference_mode: some subspace bases (e.g. the unembedding rows in EXPERIMENT C)
# are derived from live model parameters, and the recipient has requires_grad back on after
# CELL 9. Without this, every ablation call would build and retain an autograd graph.
@torch.no_grad()
def confer_first_T(T, idxs=None, batch=ARITH_BATCH):
    """First-token conferral on `idxs` (default: the unsolvable bin) with the written vector
    passed through T. T=lambda v: v reproduces the headline conferral."""
    idxs = list(unsolv) if idxs is None else list(idxs)
    probs = [evalp[j] for j in idxs]
    v = T(write_vec(X9e[idxs]))
    tops = graft_first_tokens(probs, v, batch=batch)
    return [tops[k] == probs[k]["tok"] for k in range(len(probs))]

@torch.no_grad()
def confer_full_T(T, idxs=None, batch=ARITH_BATCH):
    """Free-generation FULL-ANSWER conferral under the same transform."""
    idxs = list(unsolv) if idxs is None else list(idxs)
    probs = [evalp[j] for j in idxs]
    v = T(write_vec(X9e[idxs]))
    _, ints = graft_generations(probs, v, batch=batch)
    return [ints[k] is not None and abs(ints[k] - probs[k]["ans"]) < 0.5 for k in range(len(probs))]

def tok_digit(t):
    """The digit character a token decodes to (handles the leading-space variant), or None."""
    s = tokenizer.decode([int(t)]).strip()
    return s[0] if (len(s) and s[0].isdigit()) else None

# sanity: T = identity must reproduce the CELL 10 headline
_id_first = confer_first_T(lambda v: v)
BASE_FIRST_BOOLS = _id_first
BASE_FIRST = float(np.mean(_id_first))
NATIVE_FIRST = float(np.mean([two_top[j] == evalp[j]["tok"] for j in unsolv])) if len(unsolv) else float("nan")
print(f"conventions ready | ||mu_native|| = {MU_NATIVE.norm().item():.2f} | "
      f"baseline first-token conferral on the unsolvable bin (n={len(unsolv)}) = {BASE_FIRST:.4f} | "
      f"native floor = {NATIVE_FIRST:.4f}")
print(SINGLE_SEED_NOTE)

conventions ready | ||mu_native|| = 248.81 | baseline first-token conferral on the unsolvable bin (n=634) = 0.9038 | native floor = 0.0000
SINGLE SEED (task_maps[0]); the 5-seed spread for the same quantity is in RESULTS['arithmetic']['task_unsolv_full_per_seed']


In [14]:
# === CELL 11: INLP answer subspace A (needed by C, D and E) ===
# Iteratively null the directions in the WRITTEN vector f(h) that linearly decode the first
# answer digit. The subspace is fitted on the TRAIN items (disjoint from the unsolvable bin we
# score on), so nothing is fitted on the evaluation bin.
# Unlike the base cell this one does NOT stop at collapse: it keeps going to INLP_TARGET_RANKS
# (540) and snapshots exact-rank bases at 360 and 540 for EXPERIMENT D.
if RUN_INLP and len(unsolv):
    base_fn = map_task(MAP_SEED)                        # the exact vector CELL 10 grafts
    d_recip = int(X2t.shape[1])
    Ttr = base_fn(X9t).detach()                          # f(h) on the TRAIN items
    ytr = torch.tensor([fd(p["ans"]) for p in train], device=DEVICE)

    def _proj_out(X, Q): return X if Q is None else X - (X @ Q) @ Q.T

    print(f"  INLP baseline (un-erased) first-token conferral = {BASE_FIRST:.3f}; "
          f"native floor = {NATIVE_FIRST:.3f}", flush=True)

    blocks, Q = [], None
    inlp_curve, INLP_SNAP = [], {}
    collapse_rank = collapse_rank_strict = None
    inlp_converged_at = None
    for rnd in range(INLP_ROUNDS):
        Pw, _mu = fit_probe_w(_proj_out(Ttr, Q), ytr)
        Pc = Pw - Pw.mean(1, keepdim=True)              # drop softmax's shift-invariant direction
        D = Pc if Q is None else Pc - Q @ (Q.T @ Pc)
        Un, Sn, _ = torch.linalg.svd(D, full_matrices=False)
        keepd = Un[:, Sn > Sn.max() * 1e-3]
        if keepd.shape[1] == 0:
            inlp_converged_at = int(Q.shape[1]) if Q is not None else 0
            print(f"  INLP converged at round {rnd} (no discriminative directions left), "
                  f"rank {inlp_converged_at}"); break
        blocks.append(keepd)
        Q = torch.linalg.qr(torch.cat(blocks, 1)).Q     # QR keeps span(Q[:, :k]) == span of the
        r = int(Q.shape[1])                             # first k accumulated directions, so the
        for tr in INLP_TARGET_RANKS:                    # exact-rank snapshots below are valid.
            if r >= tr and tr not in INLP_SNAP:
                INLP_SNAP[tr] = Q[:, :tr].contiguous()
        if (rnd + 1) % INLP_CHECK_EVERY == 0 or rnd == INLP_ROUNDS - 1:
            Qr = rand_orth(d_recip, r, seed=0)
            acc = float(np.mean(confer_first_T(lambda v: ablate_zero(v, Q))))
            accm = float(np.mean(confer_first_T(lambda v: ablate_mean(v, Q))))
            accr = float(np.mean(confer_first_T(lambda v: ablate_zero(v, Qr))))
            inlp_curve.append({"rank": r, "rank_over_dmodel": round(r / d_recip, 4),
                               "answer_erased_zero": round(acc, 4),
                               "answer_erased_mean": round(accm, 4),
                               "random_erased_zero": round(accr, 4)})
            print(f"  rank {r:4d} ({r/d_recip:.3f} of d_model): zero-abl={acc:.3f}  "
                  f"mean-abl={accm:.3f}  random={accr:.3f}", flush=True)
            if collapse_rank is None and acc <= COLLAPSE_FRAC * BASE_FIRST: collapse_rank = r
            if collapse_rank_strict is None and acc <= NATIVE_FIRST + 0.02: collapse_rank_strict = r
        if r >= max(INLP_TARGET_RANKS):
            print(f"  reached target rank {r} >= {max(INLP_TARGET_RANKS)}; stopping INLP."); break

    assert Q is not None, ("INLP found no discriminative answer directions at round 0 — the task "
                           "map output carries no linearly decodable first digit at all.")
    INLP_Q = Q                                          # the converged / target-rank subspace
    INLP_RANK = int(Q.shape[1])
    for tr in INLP_TARGET_RANKS:                        # if INLP converged below a target rank,
        if tr not in INLP_SNAP: INLP_SNAP[tr] = Q       # fall back to the full converged basis
    print(f"  INLP final rank {INLP_RANK}; snapshots at {sorted(INLP_SNAP.keys())} "
          f"(actual dims: {[INLP_SNAP[k].shape[1] for k in sorted(INLP_SNAP)]})")

    RESULTS["inlp"] = {
        "d_recipient": d_recip,
        "baseline_task_first": fmt(wilson_bools(BASE_FIRST_BOOLS)),
        "native_floor_first": round(NATIVE_FIRST, 4),
        "final_rank": INLP_RANK,
        "final_rank_over_dmodel": round(INLP_RANK / d_recip, 4),
        "converged_at_rank": inlp_converged_at,
        "snapshot_ranks": {str(k): int(v.shape[1]) for k, v in sorted(INLP_SNAP.items())},
        "collapse_frac_threshold": COLLAPSE_FRAC,
        "collapse_rank": collapse_rank,
        "collapse_rank_over_dmodel": (round(collapse_rank / d_recip, 4) if collapse_rank else None),
        "collapse_rank_to_native_floor": collapse_rank_strict,
        "erased_at_final_rank_zero": fmt(wilson_bools(confer_first_T(lambda v: ablate_zero(v, INLP_Q)))),
        "erased_at_final_rank_mean": fmt(wilson_bools(confer_first_T(lambda v: ablate_mean(v, INLP_Q)))),
        "random_at_final_rank_zero": fmt(wilson_bools(confer_first_T(
            lambda v: ablate_zero(v, rand_orth(d_recip, INLP_RANK, seed=0))))),
        "curve": inlp_curve,
        "_seed": SINGLE_SEED_NOTE,
    }
    print("INLP answer-subspace erasure:", json.dumps(RESULTS["inlp"], indent=2))
else:
    INLP_Q, INLP_RANK, INLP_SNAP, inlp_curve = None, 0, {}, []
    print("INLP skipped (RUN_INLP=False or empty unsolvable bin). C/D/E will skip the "
          "INLP-dependent comparisons and still run their probe-independent parts.")

  INLP baseline (un-erased) first-token conferral = 0.904; native floor = 0.000
  rank   36 (0.016 of d_model): zero-abl=0.748  mean-abl=0.757  random=0.905
  rank   72 (0.031 of d_model): zero-abl=0.565  mean-abl=0.569  random=0.896
  rank  108 (0.047 of d_model): zero-abl=0.479  mean-abl=0.498  random=0.905
  rank  144 (0.062 of d_model): zero-abl=0.421  mean-abl=0.435  random=0.907
  rank  180 (0.078 of d_model): zero-abl=0.396  mean-abl=0.412  random=0.893
  rank  216 (0.094 of d_model): zero-abl=0.364  mean-abl=0.371  random=0.894
  rank  252 (0.109 of d_model): zero-abl=0.334  mean-abl=0.345  random=0.902
  rank  288 (0.125 of d_model): zero-abl=0.246  mean-abl=0.243  random=0.904
  rank  324 (0.141 of d_model): zero-abl=0.229  mean-abl=0.232  random=0.907
  rank  360 (0.156 of d_model): zero-abl=0.219  mean-abl=0.219  random=0.905
  rank  396 (0.172 of d_model): zero-abl=0.246  mean-abl=0.232  random=0.904
  rank  432 (0.188 of d_model): zero-abl=0.240  mean-abl=0.244  random=0.

In [15]:
# === EXPERIMENT A: PROMPT-MISMATCH ADAPTIVITY ("off-by-one") — HIGHEST PRIORITY ===
# Does the stitch deliver a STALE PRECOMPUTED VALUE, or something the recipient ADAPTS to the
# problem in front of it? Probe-free, ablation-free, subspace-free.
#
#   P  = (a, b, c)          -> donor state h_P at L9_SINGLE, last prompt token   (= X9e[i])
#   P' = (a, b, c + delta)  -> answer y' = y + delta
#   The RECIPIENT reads the prompt for P' and gets f(h_P) grafted at L2_SINGLE.
#   Three-way outcome: emits y (donor's stale answer) / y' (correct for the prompt on screen) /
#   neither.
#
# NOTE (this is the manipulation, not a flaw): the map was only ever trained on MATCHED
# donor/recipient pairs, so a mismatched prompt is off-distribution FOR THE MAP. That is exactly
# the point — the MATCHED control below re-runs the identical pipeline with the donor looking at
# the same problem the recipient reads, isolating "which problem the donor looked at" as the only
# manipulated variable.
#
# DELTA SELECTION (critical): delta is chosen PER PROBLEM so that first_digit(y) != first_digit(y')
# — otherwise first-token scoring cannot tell the two hypotheses apart — AND so that c and c+delta
# tokenize to the SAME NUMBER OF TOKENS, so the graft position is unambiguous. Problems with no
# valid delta are DROPPED and counted. The named candidate set (+/-1, +/-10, +/-37, +/-100) is a
# subset of the search below: we scan every |delta| <= A_MAX_ABS_DELTA in increasing order and
# take the first that qualifies, which returns the same delta or a smaller one. That is not a
# cosmetic change — offline on the same generator/seeds, the named set alone keeps 46% of the
# bin while the exhaustive scan keeps 76% (n ~= 485 of 638). c' is held inside [1, 99], the
# generator's own range, so P' stays in-distribution for BOTH the recipient prompt and the donor
# state used by the MATCHED control. Problems whose 99-wide reachable answer window never
# crosses a leading-digit boundary (e.g. a*b = 800, c in [1,99] -> y in [801,899]) are exactly
# the ones that must be dropped; that is the whole drop reason offline.
if RUN_A and len(unsolv):
    _EXPR_RE = _re.compile(r"^(\d+) \* (\d+) \+ (\d+)$")
    DELTAS = [d for k in range(1, A_MAX_ABS_DELTA + 1) for d in (k, -k)]
    _NTOK_CACHE = {}
    def _ntok(x):
        if x not in _NTOK_CACHE:
            _NTOK_CACHE[x] = len(tokenizer(str(x), add_special_tokens=False).input_ids)
        return _NTOK_CACHE[x]

    A_items, A_dropped = [], 0
    for i in unsolv:
        mm = _EXPR_RE.match(evalp[i]["expr"])
        if mm is None: A_dropped += 1; continue
        a, b, c = int(mm.group(1)), int(mm.group(2)), int(mm.group(3))
        y = a * b + c
        assert y == evalp[i]["ans"], "expr/ans mismatch — the generator format changed"
        pick = None
        for dlt in DELTAS:
            c2 = c + dlt
            if not (1 <= c2 <= 99): continue                 # keep P' inside the generator's range
            if _ntok(c2) != _ntok(c): continue               # SAME token count -> graft position
            y2 = a * b + c2                                  #   is unambiguous
            if fd(y2) == fd(y): continue                     # must FLIP the leading digit
            ids2, tok2 = _aencode(tokenizer, f"{a} * {b} + {c2}", y2)
            if ids2 is None or tok2 == evalp[i]["tok"]: continue   # first tokens must differ
            pick = dict(idx=i, a=a, b=b, c=c, c2=c2, delta=dlt, y=y, ans=y2, y2=y2,
                        expr=f"{a} * {b} + {c2}", ids=ids2, tok=tok2,
                        stale_tok=evalp[i]["tok"],
                        same_prompt_len=bool(ids2.numel() == evalp[i]["ids"].numel()))
            break
        if pick is None: A_dropped += 1
        else: A_items.append(pick)
    nA = len(A_items)
    print(f"A: kept {nA} / {len(unsolv)} unsolvable problems ({A_dropped} dropped: no delta in "
          f"[-{A_MAX_ABS_DELTA},{A_MAX_ABS_DELTA}] flips the leading digit at equal token count)")
    assert nA > 0, "EXPERIMENT A: no problem admitted a valid delta — widen A_MAX_ABS_DELTA."
    print("   same-prompt-length rate:",
          round(float(np.mean([p["same_prompt_len"] for p in A_items])), 4))

    # ---- the four conditions ----
    src = [p["idx"] for p in A_items]
    V_MISMATCH = write_vec(X9e[src])                                   # donor looked at P
    X9A_d, A_donor_top = states_and_top(model_9b, [L9_SINGLE],
                                        [p["ids"] for p in A_items], batch=DONOR_BATCH)
    X9A = X9A_d[L9_SINGLE]
    V_MATCHED = write_vec(X9A)                                         # donor looked at P'
    _rngA = np.random.default_rng(A_SHUFFLE_SEED)
    shuf_src = []
    for k in range(nA):
        while True:
            s = int(_rngA.integers(0, len(evalp)))
            if s != src[k]: break
        shuf_src.append(s)
    V_SHUFFLED = write_vec(X9e[shuf_src])                              # donor looked at neither

    A_top_native   = graft_first_tokens(A_items, None)
    A_top_mismatch = graft_first_tokens(A_items, V_MISMATCH)
    A_top_matched  = graft_first_tokens(A_items, V_MATCHED)
    A_top_shuffled = graft_first_tokens(A_items, V_SHUFFLED)

    # RE-FILTER: P' must ALSO be in the unsolvable bin, else the y'-bin is contaminated by the
    # recipient's own native ability on the perturbed problem.
    A_BIN = [k for k in range(nA) if A_top_native[k] != A_items[k]["tok"]]
    print(f"   P' also natively unsolvable: n={len(A_BIN)} / {nA} "
          f"(recipient natively solves {nA-len(A_BIN)} of the perturbed prompts)")
    assert len(A_BIN) > 0, "EXPERIMENT A: the recipient natively solves every perturbed prompt."
    A_donor_solves_Pp = float(np.mean([A_donor_top[k] == A_items[k]["tok"] for k in A_BIN]))

    def _cls_first(tops):
        out = {"adapted": [], "stale": [], "neither": [], "neither_leaddigit_is_yP": []}
        for k in A_BIN:
            t, it = tops[k], A_items[k]
            ad = (t == it["tok"]); st = (t == it["stale_tok"])
            out["adapted"].append(ad); out["stale"].append(st)
            out["neither"].append(not (ad or st))
            if not (ad or st):
                out["neither_leaddigit_is_yP"].append(tok_digit(t) == str(fd(it["y"])))
        return out
    def _pack_first(tops):
        c = _cls_first(tops)
        d = {"emits_y_prime_ADAPTED": fmt(wilson_bools(c["adapted"])),
             "emits_y_STALE":         fmt(wilson_bools(c["stale"])),
             "NEITHER":               fmt(wilson_bools(c["neither"])),
             "n": len(A_BIN)}
        d["neither_bucket_leaddigit_matches_yP"] = (
            fmt(wilson_bools(c["neither_leaddigit_is_yP"])) if c["neither_leaddigit_is_yP"] else "n/a")
        d["_neither_bucket_n"] = int(sum(c["neither"]))
        return d

    A_out = {
        "_what": ("recipient reads P'=(a,b,c+delta); the grafted vector is f(donor state on X). "
                  "MISMATCH: X=P (stale). MATCHED: X=P' (pipeline control). SHUFFLED: X=unrelated. "
                  "NATIVE: no graft."),
        "n_unsolvable_bin": len(unsolv), "n_with_valid_delta": nA, "n_dropped_no_delta": A_dropped,
        "n_scored_Pprime_also_unsolvable": len(A_BIN),
        "same_prompt_token_length_rate": round(float(np.mean([p["same_prompt_len"] for p in A_items])), 4),
        "donor_solves_Pprime_rate_on_bin": round(A_donor_solves_Pp, 4),
        "delta_distribution_kept": dict(sorted(collections.Counter(
            [p["delta"] for p in A_items]).items(), key=lambda kv: abs(kv[0]))),
        "delta_distribution_scored_bin": dict(sorted(collections.Counter(
            [A_items[k]["delta"] for k in A_BIN]).items(), key=lambda kv: abs(kv[0]))),
        "abs_delta_median": float(np.median([abs(A_items[k]["delta"]) for k in A_BIN])),
        "first_token": {
            "MISMATCH_stale_graft": _pack_first(A_top_mismatch),
            "MATCHED_graft_control": _pack_first(A_top_matched),
            "SHUFFLED_control":      _pack_first(A_top_shuffled),
            "NATIVE_no_graft":       _pack_first(A_top_native),
        },
        "_reference_headline_matched_on_original_bin":
            RESULTS.get("arithmetic", {}).get("task_unsolv_first_pooled", round(BASE_FIRST, 4)),
        "_seed": SINGLE_SEED_NOTE,
    }

    if A_FULL_ANSWER:
        sub = [A_items[k] for k in A_BIN]
        def _cls_full(vecs):
            _, ints = graft_generations(sub, vecs)
            ad = [ints[k] is not None and abs(ints[k] - sub[k]["y2"]) < 0.5 for k in range(len(sub))]
            st = [ints[k] is not None and abs(ints[k] - sub[k]["y"])  < 0.5 for k in range(len(sub))]
            ne = [not (ad[k] or st[k]) for k in range(len(sub))]
            lead = [fd(ints[k]) == fd(sub[k]["y"]) for k in range(len(sub))
                    if ne[k] and ints[k] is not None]
            return {"emits_y_prime_ADAPTED": fmt(wilson_bools(ad)),
                    "emits_y_STALE": fmt(wilson_bools(st)),
                    "NEITHER": fmt(wilson_bools(ne)),
                    "neither_bucket_leaddigit_matches_yP": (fmt(wilson_bools(lead)) if lead else "n/a"),
                    "n": len(sub)}
        A_out["full_answer"] = {
            "MISMATCH_stale_graft":  _cls_full(V_MISMATCH[A_BIN]),
            "MATCHED_graft_control": _cls_full(V_MATCHED[A_BIN]),
            "SHUFFLED_control":      _cls_full(V_SHUFFLED[A_BIN]),
            "NATIVE_no_graft":       _cls_full(None),
        }
        A_examples = []
        for k in A_BIN[:8]:
            A_examples.append({"P": evalp[A_items[k]["idx"]]["expr"], "y": A_items[k]["y"],
                               "P_prime": A_items[k]["expr"], "y_prime": A_items[k]["y2"],
                               "delta": A_items[k]["delta"],
                               "mismatch_first_tok": repr(tokenizer.decode([A_top_mismatch[k]])),
                               "matched_first_tok": repr(tokenizer.decode([A_top_matched[k]]))})
        A_out["examples"] = A_examples

    RESULTS["A_prompt_mismatch"] = A_out
    print("EXPERIMENT A:", json.dumps(A_out, indent=2))
else:
    print("EXPERIMENT A skipped (RUN_A=False or empty unsolvable bin).")

A: kept 555 / 634 unsolvable problems (79 dropped: no delta in [-98,98] flips the leading digit at equal token count)
   same-prompt-length rate: 1.0
   P' also natively unsolvable: n=306 / 555 (recipient natively solves 249 of the perturbed prompts)
EXPERIMENT A: {
  "_what": "recipient reads P'=(a,b,c+delta); the grafted vector is f(donor state on X). MISMATCH: X=P (stale). MATCHED: X=P' (pipeline control). SHUFFLED: X=unrelated. NATIVE: no graft.",
  "n_unsolvable_bin": 634,
  "n_with_valid_delta": 555,
  "n_dropped_no_delta": 79,
  "n_scored_Pprime_also_unsolvable": 306,
  "same_prompt_token_length_rate": 1.0,
  "donor_solves_Pprime_rate_on_bin": 0.5915,
  "delta_distribution_kept": {
    "1": 9,
    "-1": 16,
    "-2": 11,
    "2": 5,
    "3": 9,
    "-3": 13,
    "4": 9,
    "-4": 11,
    "-5": 10,
    "5": 10,
    "6": 9,
    "-6": 8,
    "-7": 7,
    "7": 5,
    "-8": 6,
    "8": 5,
    "9": 7,
    "-9": 10,
    "-10": 7,
    "10": 4,
    "11": 3,
    "-11": 3,
    "-12": 5,
  

In [16]:
# === EXPERIMENT B: RESCORING THE SHUFFLE CONTROL FOR STALE-ANSWER EMISSION ===
# CORRECTION vs the existing notebooks: shuffle_confer returns BOOLS and discards the token ids
# and text, so the shuffle generations were never saved. This cell therefore RE-RUNS the shuffle
# condition WITH OUTPUT CAPTURE. It is cheap (one forward + one generate pass, no training) but
# it is NOT a pure rescoring pass.
#
# Shuffle feeds the map the donor state for P while the recipient reads P'. We score:
#   (1) first token == first_token(y_P)  — the donor's answer to a problem the recipient NEVER SAW
#   (2) full free generation == y_P
#   (3) "correct for P'" — the existing shuffle number, for reference
#
# FILTER: pairs with first_token(y_P) == first_token(y_P') are UNINFORMATIVE and are excluded.
# EMPIRICAL NULL (required — first digits are NOT uniform, Benford-ish): pair each generation
# with the answer of a THIRD unrelated problem from the same pool, average over B_NULL_DRAWS
# draws, and report the EXCESS of the true rate over that null.
if RUN_B and len(unsolv):
    _rngB = np.random.default_rng(B_SHUFFLE_SEED)
    recip_idx = list(unsolv)
    donor_idx = []
    for j in recip_idx:                       # derangement: donor state from a DIFFERENT problem
        while True:
            s = int(_rngB.integers(0, len(evalp)))
            if s != j: break
        donor_idx.append(s)
    keptB = [k for k in range(len(recip_idx))
             if evalp[donor_idx[k]]["tok"] != evalp[recip_idx[k]]["tok"]]
    nB = len(keptB)
    print(f"B: {nB} / {len(recip_idx)} shuffle pairs kept "
          f"({len(recip_idx)-nB} dropped: first_token(y_P) == first_token(y_P'))")
    assert nB > 0, "EXPERIMENT B: every shuffle pair shares a first token."

    B_probs   = [evalp[recip_idx[k]] for k in keptB]
    B_vecs    = write_vec(X9e[[donor_idx[k] for k in keptB]])
    B_tops    = graft_first_tokens(B_probs, B_vecs)
    B_texts, B_ints = graft_generations(B_probs, B_vecs)
    d_tok = [evalp[donor_idx[k]]["tok"] for k in keptB]
    d_ans = [evalp[donor_idx[k]]["ans"] for k in keptB]
    r_tok = [evalp[recip_idx[k]]["tok"] for k in keptB]
    r_ans = [evalp[recip_idx[k]]["ans"] for k in keptB]

    first_is_donor = [B_tops[i] == d_tok[i] for i in range(nB)]
    full_is_donor  = [B_ints[i] is not None and abs(B_ints[i] - d_ans[i]) < 0.5 for i in range(nB)]
    first_is_recip = [B_tops[i] == r_tok[i] for i in range(nB)]
    full_is_recip  = [B_ints[i] is not None and abs(B_ints[i] - r_ans[i]) < 0.5 for i in range(nB)]

    # ---- empirical null: a THIRD unrelated problem, same pool, same exclusion filter ----
    null_first, null_full = [], []
    for d in range(B_NULL_DRAWS):
        rg = np.random.default_rng(90000 + d)
        nf, nu = [], []
        for i in range(nB):
            k = keptB[i]; t = None
            for _ in range(200):
                cand = int(rg.integers(0, len(evalp)))
                if (cand != donor_idx[k] and cand != recip_idx[k]
                        and evalp[cand]["tok"] != r_tok[i]):
                    t = cand; break
            if t is None: continue
            nf.append(B_tops[i] == evalp[t]["tok"])
            nu.append(B_ints[i] is not None and abs(B_ints[i] - evalp[t]["ans"]) < 0.5)
        null_first.append(float(np.mean(nf))); null_full.append(float(np.mean(nu)))
    nf_m, nf_s = float(np.mean(null_first)), float(np.std(null_first, ddof=1))
    nu_m, nu_s = float(np.mean(null_full)),  float(np.std(null_full,  ddof=1))

    B_out = {
        "_what": ("shuffle RE-RUN with output capture: the recipient reads P' while the map is fed "
                  "the donor state for an unrelated P. Does it emit y_P, the answer to a problem it "
                  "never saw?"),
        "n_pairs_kept": nB, "n_pairs_dropped_same_first_token": len(recip_idx) - nB,
        "first_token_is_donor_answer_y_P":  fmt(wilson_bools(first_is_donor)),
        "full_generation_is_donor_answer_y_P": fmt(wilson_bools(full_is_donor)),
        "first_token_correct_for_Pprime":  fmt(wilson_bools(first_is_recip)),
        "full_generation_correct_for_Pprime": fmt(wilson_bools(full_is_recip)),
        "empirical_null_draws": B_NULL_DRAWS,
        "null_first_token_match_mean": round(nf_m, 4),
        "null_first_token_match_sd": round(nf_s, 4),
        "null_full_match_mean": round(nu_m, 4),
        "null_full_match_sd": round(nu_s, 4),
        "EXCESS_first_token_over_null": round(float(np.mean(first_is_donor)) - nf_m, 4),
        "EXCESS_full_over_null": round(float(np.mean(full_is_donor)) - nu_m, 4),
        "excess_first_in_null_sds": (round((float(np.mean(first_is_donor)) - nf_m) / nf_s, 2)
                                     if nf_s > 0 else None),
        "_null_note": ("the null pairs each captured generation with a THIRD unrelated problem "
                       "drawn from the same pool under the same first-token exclusion, so it "
                       "absorbs the non-uniform (Benford-ish) leading-digit distribution."),
        "_ref_base_shuffle_first": RESULTS.get("arithmetic", {}).get("shuffle_unsolv_first", "n/a"),
        "emitted_first_digit_histogram": dict(sorted(collections.Counter(
            [tok_digit(t) for t in B_tops]).items(), key=lambda kv: str(kv[0]))),
        "gold_yP_first_digit_histogram": dict(sorted(collections.Counter(
            [str(fd(v)) for v in d_ans]).items())),
        "examples": [{"P_donor": evalp[donor_idx[keptB[i]]]["expr"], "y_P": d_ans[i],
                      "P_prime_read": B_probs[i]["expr"], "y_Pprime": r_ans[i],
                      "generated": B_texts[i][:24]} for i in range(min(8, nB))],
        "_seed": SINGLE_SEED_NOTE,
    }
    RESULTS["B_shuffle_rescored"] = B_out
    print("EXPERIMENT B:", json.dumps(B_out, indent=2))
else:
    print("EXPERIMENT B skipped (RUN_B=False or empty unsolvable bin).")

B: 564 / 634 shuffle pairs kept (70 dropped: first_token(y_P) == first_token(y_P'))
EXPERIMENT B: {
  "_what": "shuffle RE-RUN with output capture: the recipient reads P' while the map is fed the donor state for an unrelated P. Does it emit y_P, the answer to a problem it never saw?",
  "n_pairs_kept": 564,
  "n_pairs_dropped_same_first_token": 70,
  "first_token_is_donor_answer_y_P": "0.931 [0.907, 0.949]",
  "full_generation_is_donor_answer_y_P": "0.005 [0.002, 0.016]",
  "first_token_correct_for_Pprime": "0.014 [0.007, 0.028]",
  "full_generation_correct_for_Pprime": "0.007 [0.003, 0.018]",
  "empirical_null_draws": 20,
  "null_first_token_match_mean": 0.2016,
  "null_first_token_match_sd": 0.0163,
  "null_full_match_mean": 0.0014,
  "null_full_match_sd": 0.0017,
  "EXCESS_first_token_over_null": 0.7293,
  "EXCESS_full_over_null": 0.0039,
  "excess_first_in_null_sds": 44.79,
  "_null_note": "the null pairs each captured generation with a THIRD unrelated problem drawn from the same p

In [17]:
# === EXPERIMENT C: UNEMBEDDING-SPAN ABLATION (probe-independent answer subspace) ===
# Makes the causal claim independent of the INLP probe used to define the answer subspace: the
# subspace here is read straight off the RECIPIENT's own output head, fitted to nothing.
#
# 1. Take the recipient's unembedding rows for the tokens that can appear as the FIRST answer
#    token — the digit tokens ACTUALLY OBSERVED (gold targets in the eval set + tokens actually
#    emitted under the stitch), so leading-space variants are caught empirically.
# 2. Gemma-2 applies a final RMSNorm with a learned elementwise gain gamma BEFORE the unembedding,
#    so the effective residual-space direction for token t is gamma * W_U[t]. We use the
#    GAIN-SCALED rows, and extract gamma empirically (RMSNorm(100*e_j)[j]/sqrt(d)) so the result
#    does not depend on whether this transformers build stores the gain as `w` or as `1 + w`
#    (Gemma stores 1 + w). Gemma-2 also TIES input/output embeddings, so W_U is the embedding
#    matrix — the orientation assert below checks that. (Gemma-2's final logit softcapping at 30.0
#    does not change directions, but it WILL confuse any magnitude sanity-check.)
# 3. Mean-centre the rows and orthonormalise -> basis B (d_recip x <= k-1; mean-centering removes
#    one dimension, which is why rank <= 9 for 10 digit tokens).
# 4. Ablate B from f(h), ZERO and MEAN, and measure first-token conferral on the unsolvable bin.
# PARTIAL collapse is the expectation here (the answer is stored redundantly; ~9 dims need not
# capture all of it) — that is informative, not a failure.
if RUN_C and len(unsolv):
    C_probs = [evalp[j] for j in unsolv]
    C_tops  = graft_first_tokens(C_probs, write_vec(X9e[unsolv]))
    gold_toks    = sorted({p["tok"] for p in evalp})
    emitted_toks = sorted({t for t in set(C_tops) if tok_digit(t) is not None})
    ANS_TOKS = sorted(set(gold_toks) | set(emitted_toks))
    print("C: answer tokens (id -> decoded):",
          {int(t): repr(tokenizer.decode([int(t)])) for t in ANS_TOKS})
    print(f"   {len(gold_toks)} gold first-answer tokens, {len(emitted_toks)} digit-bearing "
          f"tokens actually emitted under the stitch, {len(ANS_TOKS)} in the union")

    lm_w = model_2b.lm_head.weight.detach()      # detach: the recipient has requires_grad back
    assert lm_w.shape[1] == D_RECIP, ("unembedding orientation: expected (vocab, d_model), got "
                                      f"{tuple(lm_w.shape)} with d_model={D_RECIP}")
    TIED_EMB = (lm_w.data_ptr() == model_2b.model.embed_tokens.weight.data_ptr())
    _nodigit = [int(t) for t in ANS_TOKS if tok_digit(t) is None]
    if _nodigit:
        print("   WARNING: these observed 'answer' tokens decode to no leading digit:",
              {int(t): repr(tokenizer.decode([int(t)])) for t in _nodigit})
    with torch.no_grad():
        _E = torch.eye(D_RECIP, device=DEVICE, dtype=torch.float32) * 100.0
        GAIN = (torch.diagonal(model_2b.model.norm(_E).float()) / math.sqrt(D_RECIP)).contiguous()
    del _E
    _gain_decl = (1.0 + model_2b.model.norm.weight.detach().float()).to(DEVICE)
    _gd = float((GAIN - _gain_decl).abs().max().item())
    print(f"   tied embeddings: {TIED_EMB} | RMSNorm gain: empirical vs (1+w) max|diff| = {_gd:.4g} "
          f"| mean gain {GAIN.mean().item():.3f}")

    rows   = GAIN.unsqueeze(0) * lm_w[ANS_TOKS].float().to(DEVICE)     # (k, d) gain-scaled
    rows_c = rows - rows.mean(0, keepdim=True)                          # mean-centre
    B_UNEMB = orth_basis(rows_c.T)                                      # (d, rank <= k-1)
    RB = int(B_UNEMB.shape[1])
    print(f"   unembedding answer basis B: {len(ANS_TOKS)} rows -> rank {RB}")

    C_zero = confer_first_T(lambda v: ablate_zero(v, B_UNEMB))
    C_mean = confer_first_T(lambda v: ablate_mean(v, B_UNEMB))
    C_zero_full = confer_full_T(lambda v: ablate_zero(v, B_UNEMB))
    C_mean_full = confer_full_T(lambda v: ablate_mean(v, B_UNEMB))

    rnd_zero, rnd_mean = [], []
    for s in range(C_RANDOM_SEEDS):
        Qr = rand_orth(D_RECIP, RB, seed=1000 + s)
        rnd_zero.append(float(np.mean(confer_first_T(lambda v: ablate_zero(v, Qr)))))
        rnd_mean.append(float(np.mean(confer_first_T(lambda v: ablate_mean(v, Qr)))))
    print(f"   matched-rank random control ({C_RANDOM_SEEDS} seeds): "
          f"zero {np.mean(rnd_zero):.3f}, mean {np.mean(rnd_mean):.3f}", flush=True)

    C_out = {
        "_what": "ablate the recipient's own gain-scaled unembedding span for the observed answer tokens",
        "answer_token_ids": [int(t) for t in ANS_TOKS],
        "answer_tokens_decoded": [tokenizer.decode([int(t)]) for t in ANS_TOKS],
        "n_answer_tokens": len(ANS_TOKS), "basis_rank": RB,
        "tied_input_output_embeddings": bool(TIED_EMB),
        "rmsnorm_gain_empirical_vs_1_plus_w_maxdiff": round(_gd, 8),
        "baseline_first": fmt(wilson_bools(BASE_FIRST_BOOLS)),
        "native_floor_first": round(NATIVE_FIRST, 4),
        "unembedding_ablated_first_ZERO": fmt(wilson_bools(C_zero)),
        "unembedding_ablated_first_MEAN": fmt(wilson_bools(C_mean)),
        "unembedding_ablated_full_ZERO": fmt(wilson_bools(C_zero_full)),
        "unembedding_ablated_full_MEAN": fmt(wilson_bools(C_mean_full)),
        "random_matched_rank_first_ZERO": fmt(across_seed_ci(rnd_zero)),
        "random_matched_rank_first_MEAN": fmt(across_seed_ci(rnd_mean)),
        "random_seeds": C_RANDOM_SEEDS,
        "_seed": SINGLE_SEED_NOTE,
    }
    if INLP_Q is not None:
        k9 = min(RB, INLP_RANK)
        Q9 = INLP_Q[:, :k9].contiguous()                  # the first RB INLP directions
        C_out["inlp_first_%d_dirs_first_ZERO" % k9] = fmt(wilson_bools(
            confer_first_T(lambda v: ablate_zero(v, Q9))))
        C_out["inlp_first_%d_dirs_first_MEAN" % k9] = fmt(wilson_bools(
            confer_first_T(lambda v: ablate_mean(v, Q9))))
        C_out["principal_angles_B_vs_INLP_converged_deg"] = angles_deg(B_UNEMB, INLP_Q)
        C_out["containment_B_in_INLP_converged"] = round(containment(B_UNEMB, INLP_Q), 4)
        C_out["containment_null_INLP"] = round(INLP_RANK / D_RECIP, 4)
        C_out["principal_angles_B_vs_INLP_first_%d_deg" % k9] = angles_deg(B_UNEMB, Q9)
    RESULTS["C_unembedding_span"] = C_out
    print("EXPERIMENT C:", json.dumps(C_out, indent=2))
else:
    B_UNEMB, ANS_TOKS = None, []
    print("EXPERIMENT C skipped (RUN_C=False or empty unsolvable bin).")

C: answer tokens (id -> decoded): {235274: "'1'", 235284: "'2'", 235304: "'3'", 235308: "'5'", 235310: "'4'", 235315: "'9'", 235318: "'6'", 235321: "'8'", 235324: "'7'"}
   9 gold first-answer tokens, 9 digit-bearing tokens actually emitted under the stitch, 9 in the union
   tied embeddings: True | RMSNorm gain: empirical vs (1+w) max|diff| = 4.768e-07 | mean gain 3.453
   unembedding answer basis B: 9 rows -> rank 8
   matched-rank random control (20 seeds): zero 0.904, mean 0.904
EXPERIMENT C: {
  "_what": "ablate the recipient's own gain-scaled unembedding span for the observed answer tokens",
  "answer_token_ids": [
    235274,
    235284,
    235304,
    235308,
    235310,
    235315,
    235318,
    235321,
    235324
  ],
  "answer_tokens_decoded": [
    "1",
    "2",
    "3",
    "5",
    "4",
    "9",
    "6",
    "8",
    "7"
  ],
  "n_answer_tokens": 9,
  "basis_rank": 8,
  "tied_input_output_embeddings": true,
  "rmsnorm_gain_empirical_vs_1_plus_w_maxdiff": 4.8e-07,
  "ba

In [18]:
# === EXPERIMENT D: SUFFICIENCY SPLIT ===
# CORRECTION vs the plan-as-written: `(I-P) f(h)` IS the existing INLP ablation — ablating the
# answer subspace is the same operation as grafting only the complement. So the NEW content here
# is the PROJECTION-ONLY condition. Both are implemented; D is framed as the SUFFICIENCY test:
#
#   proj_only     : graft P f(h) ALONE, everything else ZEROED
#   proj_meanfill : graft P f(h) with the complement filled from mu_native (on-distribution)
#   comp_only     : graft (I-P) f(h) alone  == the existing zero-ablation
#   comp_meanfill : (I-P) f(h) + P mu_native == the existing mean-ablation
#
# NORM CONTROL (important): the components have different magnitudes than f(h), and the graft
# OVERWRITES the residual, so magnitude matters on its own. Each component is therefore run BOTH
# at its natural norm AND rescaled per-example to ||f(h)||. Both are reported.
if RUN_D and len(unsolv):
    D_SUBSPACES = {}
    if INLP_Q is not None:
        for r in D_SUBSPACE_RANKS:
            Qs = INLP_SNAP.get(r, INLP_Q[:, :min(r, INLP_RANK)]).contiguous()
            D_SUBSPACES["INLP_rank%d" % int(Qs.shape[1])] = Qs
    if B_UNEMB is not None:
        D_SUBSPACES["UNEMB_rank%d" % int(B_UNEMB.shape[1])] = B_UNEMB
    assert D_SUBSPACES, "EXPERIMENT D needs at least one subspace (run CELL 11 and/or EXPERIMENT C)"
    print("D: subspaces ->", {k: int(v.shape[1]) for k, v in D_SUBSPACES.items()})

    # norm bookkeeping: how big is each component relative to f(h)?
    Vfull = write_vec(X9e[unsolv])
    nrm_full = float(Vfull.norm(dim=1).mean().item())
    nrm_native = float(X2e[unsolv].norm(dim=1).mean().item())

    D_out = {
        "_what": ("sufficiency: is the answer subspace ENOUGH on its own? proj_only is the new "
                  "condition; comp_only IS the existing INLP ablation, reported for completeness."),
        "mean_norm_f_h": round(nrm_full, 3),
        "mean_norm_native_recipient_state": round(nrm_native, 3),
        "baseline_first": fmt(wilson_bools(BASE_FIRST_BOOLS)),
        "native_floor_first": round(NATIVE_FIRST, 4),
        "_seed": SINGLE_SEED_NOTE, "subspaces": {},
    }
    for name, Q in D_SUBSPACES.items():
        pj = _proj(Vfull, Q)
        cp = Vfull - pj
        ent = {
            "rank": int(Q.shape[1]),
            "mean_norm_projection": round(float(pj.norm(dim=1).mean().item()), 3),
            "mean_norm_complement": round(float(cp.norm(dim=1).mean().item()), 3),
            "mean_energy_fraction_in_subspace": round(float(
                ((pj.norm(dim=1) ** 2) / (Vfull.norm(dim=1) ** 2 + 1e-12)).mean().item()), 4),
        }
        conds = {
            "proj_only_natural":      lambda v, Q=Q: project_only(v, Q),
            "proj_only_rescaled":     lambda v, Q=Q: rescale_to(project_only(v, Q), v),
            "proj_meanfill_natural":  lambda v, Q=Q: project_only_meanfill(v, Q),
            "proj_meanfill_rescaled": lambda v, Q=Q: rescale_to(project_only_meanfill(v, Q), v),
            "comp_only_natural":      lambda v, Q=Q: ablate_zero(v, Q),
            "comp_only_rescaled":     lambda v, Q=Q: rescale_to(ablate_zero(v, Q), v),
            "comp_meanfill_natural":  lambda v, Q=Q: ablate_mean(v, Q),
            "comp_meanfill_rescaled": lambda v, Q=Q: rescale_to(ablate_mean(v, Q), v),
        }
        for cname, T in conds.items():
            ent[cname + "__first"] = fmt(wilson_bools(confer_first_T(T)))
            print(f"   {name:>16s} {cname:>24s} first = {ent[cname+'__first']}", flush=True)
        if D_FULL_ANSWER:
            for cname in ["proj_only_natural", "proj_meanfill_natural",
                          "comp_only_natural", "comp_meanfill_natural"]:
                ent[cname + "__full"] = fmt(wilson_bools(confer_full_T(conds[cname])))
                print(f"   {name:>16s} {cname:>24s} full  = {ent[cname+'__full']}", flush=True)
        ent["_note_comp_only_natural"] = ("identical operation to the existing INLP zero-ablation; "
                                          "comp_meanfill_natural is its mean-ablation counterpart")
        D_out["subspaces"][name] = ent
    if D_FULL_ANSWER:
        D_out["baseline_full"] = fmt(wilson_bools(confer_full_T(lambda v: v)))
    RESULTS["D_sufficiency_split"] = D_out
    print("EXPERIMENT D:", json.dumps(D_out, indent=2))
else:
    print("EXPERIMENT D skipped (RUN_D=False or empty unsolvable bin).")

D: subspaces -> {'INLP_rank540': 540, 'INLP_rank360': 360, 'UNEMB_rank8': 8}
       INLP_rank540        proj_only_natural first = 0.896 [0.870, 0.917]
       INLP_rank540       proj_only_rescaled first = 0.888 [0.861, 0.910]
       INLP_rank540    proj_meanfill_natural first = 0.896 [0.870, 0.917]
       INLP_rank540   proj_meanfill_rescaled first = 0.885 [0.858, 0.907]
       INLP_rank540        comp_only_natural first = 0.202 [0.172, 0.235]
       INLP_rank540       comp_only_rescaled first = 0.167 [0.140, 0.198]
       INLP_rank540    comp_meanfill_natural first = 0.196 [0.167, 0.228]
       INLP_rank540   comp_meanfill_rescaled first = 0.139 [0.114, 0.168]
       INLP_rank540        proj_only_natural full  = 0.110 [0.088, 0.137]
       INLP_rank540    proj_meanfill_natural full  = 0.120 [0.097, 0.147]
       INLP_rank540        comp_only_natural full  = 0.036 [0.024, 0.054]
       INLP_rank540    comp_meanfill_natural full  = 0.038 [0.026, 0.056]
       INLP_rank360        proj_onl

In [19]:
# === EXPERIMENT E: SVD ABLATION CURVE + CONTAINMENT ===
# Resolves a real tension: rank-16 TRUNCATION of the task map recovers ~0.875 of the ~0.887
# full-rank conferral, yet INLP needs 360-540 directions to collapse conferral.
#
# 1. SVD the learned linear map. CAREFUL WITH ORIENTATION: the code stores the map as
#    v = (h - mu9) @ W_code + b with W_code of shape (d_donor, d_recip), so the mathematical map
#    is M = W_code.T of shape (d_recip x d_donor). The columns of U in M = U S V^T are the
#    RECIPIENT-SPACE directions the map WRITES INTO. U is the relevant one.
# 2. Sweep j: ablate the TOP-j LEFT singular directions FROM the written vector,
#    f(h) <- (I - U_j U_j^T) f(h). This is the COMPLEMENT of the existing rank-truncation sweep
#    (that KEEPS the top j; this REMOVES them).
# 3. Emit overlay data for three curves: SVD ablation, INLP ablation, matched-rank random.
# CONTAINMENT: singular values of U_k^T B are cos(theta_i); report (1/k) sum cos^2 PLUS the
# per-angle degrees. NULLS ARE NOT OPTIONAL: a random k-dim subspace has expected containment
# dim(target)/d — 9/2304 ~= 0.0039 against the unembedding span, 540/2304 ~= 0.234 against the
# 540-dim INLP subspace. We LEAD with the unembedding span B (fitted to nothing) and report the
# INLP version alongside.
if RUN_E and len(unsolv):
    W_code = task_maps[MAP_SEED][0]
    M = W_code.T.float().contiguous()                 # (d_recip, d_donor) — math convention
    assert M.shape[0] == D_RECIP and M.shape[1] == D_DONOR, (
        f"map orientation wrong: M is {tuple(M.shape)}, expected ({D_RECIP}, {D_DONOR})")
    U, S, Vh = torch.linalg.svd(M, full_matrices=False)
    U = U.contiguous()                                # (d_recip, r) recipient-space WRITE dirs
    Sf = S.float()
    print(f"E: SVD of the map done. U {tuple(U.shape)} (recipient-space write directions), "
          f"S[0]={Sf[0].item():.4g}, S[15]={Sf[min(15, len(Sf)-1)].item():.4g}")

    js = [j for j in E_JS if j <= U.shape[1]]
    svd_curve = []
    for j in js:
        Uj = U[:, :j].contiguous()
        az = float(np.mean(confer_first_T(lambda v: ablate_zero(v, Uj))))
        am = float(np.mean(confer_first_T(lambda v: ablate_mean(v, Uj))))
        rz = [float(np.mean(confer_first_T(lambda v, Qr=rand_orth(D_RECIP, j, seed=2000 + s):
                                           ablate_zero(v, Qr))))
              for s in range(E_RANDOM_SEEDS)]
        svd_curve.append({
            "j": j, "j_over_dmodel": round(j / D_RECIP, 5),
            "svd_ablated_zero": round(az, 4), "svd_ablated_mean": round(am, 4),
            "random_ablated_zero_mean": round(float(np.mean(rz)), 4),
            "random_ablated_zero_sd": round(float(np.std(rz, ddof=1)) if len(rz) > 1 else 0.0, 4),
            "spectrum_energy_fraction_in_top_j": round(float(
                (Sf[:j] ** 2).sum().item() / (Sf ** 2).sum().item()), 4),
        })
        print(f"   j={j:4d}: svd-abl zero={az:.3f} mean={am:.3f} | random={np.mean(rz):.3f}", flush=True)

    E_out = {
        "_what": ("remove the TOP-j directions the map writes into (complement of the existing "
                  "rank-truncation sweep, which keeps them)"),
        "baseline_first": fmt(wilson_bools(BASE_FIRST_BOOLS)),
        "native_floor_first": round(NATIVE_FIRST, 4),
        "map_orientation": ("code stores v = (h-mu9)@W_code + b with W_code (d_donor, d_recip); "
                            "M = W_code.T; U = left singular vectors of M = recipient-space writes"),
        "singular_values_top32": [round(float(x), 5) for x in Sf[:32].tolist()],
        "svd_ablation_curve": svd_curve,
        "random_seeds_per_j": E_RANDOM_SEEDS,
        "_seed": SINGLE_SEED_NOTE,
    }

    # ---- CONTAINMENT of the map's top-k write directions in the answer subspaces ----
    cont = {}
    targets = {}
    if B_UNEMB is not None: targets["UNEMB_rank%d" % int(B_UNEMB.shape[1])] = B_UNEMB
    if INLP_Q is not None:
        targets["INLP_rank%d" % INLP_RANK] = INLP_Q
        for r in D_SUBSPACE_RANKS:
            Qs = INLP_SNAP.get(r)
            if Qs is not None and int(Qs.shape[1]) != INLP_RANK:
                targets["INLP_rank%d" % int(Qs.shape[1])] = Qs
    for tname, Qt in targets.items():
        dt = int(Qt.shape[1])
        for k in E_CONTAIN_KS:
            if k > U.shape[1]: continue
            Uk = U[:, :k].contiguous()
            nulls = [containment(rand_orth(D_RECIP, k, seed=3000 + s), Qt)
                     for s in range(E_CONTAIN_NULL_SEEDS)]
            cont["U%d_in_%s" % (k, tname)] = {
                "k": k, "target_dim": dt,
                "containment_mean_cos2": round(containment(Uk, Qt), 5),
                "analytic_null_target_dim_over_d": round(dt / D_RECIP, 5),
                "empirical_null_mean": round(float(np.mean(nulls)), 5),
                "empirical_null_ci": fmt(across_seed_ci(nulls)),
                "principal_angles_deg": angles_deg(Uk, Qt),
                "reverse_containment_target_in_Uk": round(containment(Qt, Uk), 5),
                "reverse_analytic_null_k_over_d": round(k / D_RECIP, 5),
            }
            print(f"   containment U{k} in {tname}: "
                  f"{cont['U%d_in_%s' % (k, tname)]['containment_mean_cos2']:.4f} "
                  f"(null {dt/D_RECIP:.4f})", flush=True)
    E_out["containment"] = cont
    E_out["_containment_note"] = ("LEAD with the UNEMB target: it is the recipient's own output "
                                  "head, fitted to nothing in this study. Its null is dim(B)/d.")

    # ---- overlay data: three curves on one axis (rank removed vs conferral) ----
    E_out["overlay"] = {
        "x_axis": "number of directions removed from f(h)",
        "svd_ablation": [{"rank": r["j"], "confer_zero": r["svd_ablated_zero"],
                          "confer_mean": r["svd_ablated_mean"]} for r in svd_curve],
        "random_ablation": [{"rank": r["j"], "confer_zero": r["random_ablated_zero_mean"],
                             "sd": r["random_ablated_zero_sd"]} for r in svd_curve],
        "inlp_ablation": [{"rank": r["rank"], "confer_zero": r["answer_erased_zero"],
                           "confer_mean": r["answer_erased_mean"]} for r in inlp_curve],
        "unembedding_point": (
            {"rank": int(B_UNEMB.shape[1]),
             "confer_zero": RESULTS.get("C_unembedding_span", {}).get("unembedding_ablated_first_ZERO"),
             "confer_mean": RESULTS.get("C_unembedding_span", {}).get("unembedding_ablated_first_MEAN")}
            if B_UNEMB is not None else None),
        "baseline": round(BASE_FIRST, 4), "native_floor": round(NATIVE_FIRST, 4),
    }
    RESULTS["E_svd_ablation_containment"] = E_out
    print("EXPERIMENT E:", json.dumps(E_out, indent=2))
else:
    print("EXPERIMENT E skipped (RUN_E=False or empty unsolvable bin).")

E: SVD of the map done. U (2304, 2304) (recipient-space write directions), S[0]=14.39, S[15]=6.792
   j=   1: svd-abl zero=0.707 mean=0.708 | random=0.904
   j=   2: svd-abl zero=0.352 mean=0.356 | random=0.904
   j=   4: svd-abl zero=0.270 mean=0.265 | random=0.905
   j=   8: svd-abl zero=0.222 mean=0.222 | random=0.903
   j=  16: svd-abl zero=0.155 mean=0.147 | random=0.905
   j=  32: svd-abl zero=0.115 mean=0.098 | random=0.903
   j=  64: svd-abl zero=0.131 mean=0.110 | random=0.903
   j= 128: svd-abl zero=0.139 mean=0.159 | random=0.901
   j= 256: svd-abl zero=0.142 mean=0.167 | random=0.900
   j= 512: svd-abl zero=0.104 mean=0.180 | random=0.905
   containment U8 in UNEMB_rank8: 0.2452 (null 0.0035)
   containment U16 in UNEMB_rank8: 0.2396 (null 0.0035)
   containment U32 in UNEMB_rank8: 0.1424 (null 0.0035)
   containment U8 in INLP_rank540: 0.8328 (null 0.2344)
   containment U16 in INLP_rank540: 0.8217 (null 0.2344)
   containment U32 in INLP_rank540: 0.7410 (null 0.2344)
   c

In [20]:
# === SAVE: concentrate everything and write answersubspace_followups_results.json ===
RESULTS["_meta"] = {
    "notebook": "48GB_AnswerSubspace_followups.ipynb",
    "donor": MODEL_9B, "recipient": MODEL_2B,
    "single_pair_recipientL_donorL": [L2_SINGLE, L9_SINGLE],
    "d_recipient": D_RECIP, "d_donor": D_DONOR,
    "n_train": len(train), "n_eval_donor_solved": len(evalp), "n_unsolvable_bin": len(unsolv),
    "task_map_seeds_trained": list(TASK_SEEDS), "followups_use_seed": MAP_SEED,
    "smoke_test": SMOKE_TEST,
    "flags": {k: bool(globals().get(k)) for k in
              ["RUN_CORE", "RUN_INLP", "RUN_A", "RUN_B", "RUN_C", "RUN_D", "RUN_E"]},
    "convention": ("f(h) = W h + b is the vector WRITTEN into the recipient at L2_SINGLE; every "
                   "ablation acts on that final affine output. Ablations are reported BOTH as "
                   "zero-ablation and as mean-ablation against mu_native (the recipient's mean "
                   "native L2_SINGLE state on the same problems). If they diverge, report the "
                   "mean-ablation number."),
    "single_seed_caveat": SINGLE_SEED_NOTE,
}
with open(OUT_JSON, "w") as fh:
    json.dump(RESULTS, fh, indent=2, default=str)
print("wrote", OUT_JSON, "| top-level keys:", list(RESULTS.keys()))

print("\n================ HEADLINES ================")
def _g(*ks):
    d = RESULTS
    for k in ks:
        if not isinstance(d, dict) or k not in d: return "n/a"
        d = d[k]
    return d
print("baseline first-token conferral (unsolvable bin): ", _g("arithmetic", "task_unsolv_first_pooled"))
print("A  mismatch -> y' (ADAPTED):                     ", _g("A_prompt_mismatch", "first_token", "MISMATCH_stale_graft", "emits_y_prime_ADAPTED"))
print("A  mismatch -> y  (STALE):                       ", _g("A_prompt_mismatch", "first_token", "MISMATCH_stale_graft", "emits_y_STALE"))
print("A  matched  -> y' (pipeline control ~0.89):      ", _g("A_prompt_mismatch", "first_token", "MATCHED_graft_control", "emits_y_prime_ADAPTED"))
print("B  shuffle emits donor's y_P (first token):      ", _g("B_shuffle_rescored", "first_token_is_donor_answer_y_P"))
print("B  ... EXCESS over the empirical null:           ", _g("B_shuffle_rescored", "EXCESS_first_token_over_null"))
print("C  unembedding-span ablated (zero / mean):       ", _g("C_unembedding_span", "unembedding_ablated_first_ZERO"), "/", _g("C_unembedding_span", "unembedding_ablated_first_MEAN"))
print("C  matched-rank random (zero):                   ", _g("C_unembedding_span", "random_matched_rank_first_ZERO"))
print("E  containment of U16 in the unembedding span:   ",
      {k: v.get("containment_mean_cos2") for k, v in (_g("E_svd_ablation_containment", "containment") or {}).items()
       if isinstance(v, dict) and k.startswith("U16_in_UNEMB")})
print("===========================================")

wrote answersubspace_followups_results.json | top-level keys: ['bins', 'arithmetic', 'inlp', 'A_prompt_mismatch', 'B_shuffle_rescored', 'C_unembedding_span', 'D_sufficiency_split', 'E_svd_ablation_containment', '_meta']

================ HEADLINES ================
baseline first-token conferral (unsolvable bin):  0.904 [0.878, 0.924]
A  mismatch -> y' (ADAPTED):                      0.082 [0.056, 0.118]
A  mismatch -> y  (STALE):                        0.892 [0.852, 0.922]
A  matched  -> y' (pipeline control ~0.89):       0.474 [0.419, 0.530]
B  shuffle emits donor's y_P (first token):       0.931 [0.907, 0.949]
B  ... EXCESS over the empirical null:            0.7293
C  unembedding-span ablated (zero / mean):        0.662 [0.625, 0.698] / 0.677 [0.639, 0.712]
C  matched-rank random (zero):                    0.904 [0.903, 0.905]
E  containment of U16 in the unembedding span:    {'U16_in_UNEMB_rank8': 0.23964}
